# IT Helpdesk Model Finetuning Guide

This notebook demonstrates how to finetune a generic LLM (like Llama 3) on your specific `large_error_codes.csv` dataset to make it an expert in Windows Error Codes.

## ⚠️ Prerequisites
**Note for Windows Users:** Finetuning requires significant GPU VRAM. If you have less than 16GB VRAM, we highly recommend uploading this notebook and your `dataset/large_error_codes.csv` to [Google Colab](https://colab.research.google.com/) and running it there with a T4 GPU.

In [ ]:
!pip install -q -U torch transformers peft datasets bitsandbytes trl

In [ ]:
import pandas as pd
from datasets import Dataset
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig
from trl import SFTTrainer

## 1. Load and Prepare the Dataset

In [ ]:
# Load your CSV
try:
    df = pd.read_csv("../Dataset/large_error_codes.csv")
except FileNotFoundError:
    # Fallback if running in Colab
    print("Adjusting path for Colab..")
    df = pd.read_csv("large_error_codes.csv")

# Determine columns (adjust if your CSV matches specific column names)
# Based on your file: Code, Description, Category
df.columns = [c.strip() for c in df.columns]
print(f"Loaded {len(df)} rows. Columns: {df.columns}")

# Create training prompts
def format_instruction(row):
    return f"### User: What does error code {row['Code']} mean?\n\n### Assistant: Error {row['Code']} corresponds to: {row['Description']}. It is classified under {row['Category']}."

df['text'] = df.apply(format_instruction, axis=1)
dataset = Dataset.from_pandas(df[['text']])
print(dataset[0])

## 2. Initialize Model (Quantized for Memory Efficiency)

In [ ]:
model_name = "unsloth/llama-3-8b-bnb-4bit" # Or "meta-llama/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

## 3. Define LoRA Config (Parameter Efficient Fine-Tuning)

In [ ]:
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM"
)

## 4. Train

In [ ]:
training_args = TrainingArguments(
    output_dir="./scio_finetuned",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=100, # Increase this for full training (e.g., 500-1000)
    fp16=True,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
)

trainer.train()

## 5. Save Model and Export to Ollama
After training, you save the adapters. To use in Ollama, you typically merge the model and convert to GGUF (which requires `llama.cpp`).

In [ ]:
trainer.model.save_pretrained("scio-adapter")
print("Training complete! Adapter saved.")